In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent

os.chdir(PROJECT_ROOT)

print(PROJECT_ROOT)

d:\IITG\Projects\audio_factor_disentanglement_v2


In [2]:
import gc
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt

from tqdm import tqdm

from src.utils.config_loader import load_yaml

from src.dataset.feature_loader import build_dataloader

from src.models.factorized.factorized_vae import FactorizedVAE

from src.losses.total_loss import TotalLoss

from src.trainers.memory_monitor import MemoryMonitor

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [4]:
model_cfg = load_yaml(

    PROJECT_ROOT
    /
    "configs"
    /
    "model_config.yaml"
)

train_cfg = load_yaml(

    PROJECT_ROOT
    /
    "configs"
    /
    "train_config.yaml"
)

print(model_cfg.keys())
print(train_cfg.keys())

dict_keys(['factorized_model', 'betas', 'model', 'losses'])
dict_keys(['training', 'optimizer', 'scheduler', 'gradient', 'mixed_precision', 'checkpoint', 'early_stopping', 'stage_training', 'beta_scheduler', 'latent_monitor', 'losses', 'staged_activation'])


In [5]:
model_cfg = load_yaml(
    PROJECT_ROOT /
    "configs" /
    "model_config.yaml"
)

train_cfg = load_yaml(
    PROJECT_ROOT /
    "configs" /
    "train_config.yaml"
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cpu


### **Original dataset**

In [6]:
loader = build_dataloader(
    PROJECT_ROOT,
    split="train"
)

dataset = loader.dataset

print(
    "Dataset Size:",
    len(dataset)
)

Dataset Size: 290


### **Tiny Dataset**

In [17]:
# ==================================================
# CELL 8
# Experiment Settings
# ==================================================

N_SAMPLES_LIST = [

    4,

    8
]

EPOCHS = 100

LR = 1e-4

print(
    "Experiments:",
    N_SAMPLES_LIST
)

Experiments: [4, 8]


In [18]:
# ==================================================
# CELL 7
# Dataset
# ==================================================

from torch.utils.data import (
    DataLoader,
    Subset
)

from src.dataset.feature_dataset import (
    FeatureDataset
)

from src.dataset.feature_collator import (
    FeatureCollator
)

inventory_csv = (

    PROJECT_ROOT
    /
    "data"
    /
    "metadata"
    /
    "feature_inventory_v2.csv"
)

dataset = FeatureDataset(

    inventory_csv,

    split="train"
)

print(
    "Inventory:",
    inventory_csv
)

print(
    "Dataset Size:",
    len(dataset)
)

Inventory: d:\IITG\Projects\audio_factor_disentanglement_v2\data\metadata\feature_inventory_v2.csv
Dataset Size: 290


In [19]:
# ==================================================
# CELL 7.5
# Dataset Sanity
# ==================================================

sample = dataset[0]

for k,v in sample.items():

    if torch.is_tensor(v):

        print(
            k,
            tuple(v.shape)
        )

print()

print(
    "speaker:",
    sample["speaker"]
)

print(
    "condition:",
    sample["condition"]
)

logmel (80, 37)
mr_mag_256 (129, 73)
mr_mag_512 (257, 37)
mr_mag_1024 (513, 19)
magnitude (513, 37)
if (513, 37)
modgd (513, 37)
phase_sin (513, 37)
phase_cos (513, 37)

speaker: s1
condition: clean


In [20]:
def latent_ablation_score(
    model,
    outputs,
    target_lengths,
    latent_name
):

    latents = {}

    for k,v in outputs[
        "latents"
    ].items():

        latents[k] = v.clone()

    latents[
        latent_name
    ] = torch.zeros_like(
        latents[
            latent_name
        ]
    )

    recon = model.decode(
        latents,
        target_lengths
    )

    score = {}

    for feature in [

        "logmel",
        "magnitude",
        "if"

    ]:

        score[feature] = float(

            torch.mean(

                torch.abs(

                    outputs[
                        "reconstructions"
                    ][feature]

                    -

                    recon[
                        feature
                    ]
                )
            )
        )

    return score

In [21]:
# ==================================================
# CELL 9
# Helper Functions
# ==================================================

def build_target_lengths(
    batch
):

    return {

        "logmel":
        batch["logmel"].shape[-1],

        "mr_mag_256":
        batch["mr_mag_256"].shape[-1],

        "mr_mag_512":
        batch["mr_mag_512"].shape[-1],

        "magnitude":
        batch["magnitude"].shape[-1],

        "mr_mag_1024":
        batch["mr_mag_1024"].shape[-1]
    }


def latent_ablation_score(

    model,

    outputs,

    target_lengths,

    latent_name
):

    latents = {

        k:v.clone()

        for k,v in outputs[
            "latents"
        ].items()
    }

    latents[
        latent_name
    ] = torch.zeros_like(

        latents[
            latent_name
        ]
    )

    recon = model.decode(

        latents,

        target_lengths
    )

    results = {}

    for feature in [

        "logmel",

        "magnitude",

        "if"
    ]:

        results[feature] = float(

            torch.mean(

                torch.abs(

                    outputs[
                        "reconstructions"
                    ][feature]

                    -

                    recon[
                        feature
                    ]
                )
            )
        )

    return results

In [22]:
# ==================================================
# CELL 10
# Storage
# ==================================================

all_results = {}

In [23]:
# ==================================================
# CELL 11
# Experiment Builder
# ==================================================

def run_experiment(

    n_samples,

    lr=1e-4
):

    subset_dataset = Subset(

        dataset,

        list(
            range(
                n_samples
            )
        )
    )

    subset_loader = DataLoader(

        subset_dataset,

        batch_size=1,

        shuffle=True,

        collate_fn=FeatureCollator(),

        num_workers=0,

        pin_memory=False
    )

    model = FactorizedVAE(
        model_cfg
    ).to(device)

    # ------------------------------------------
    # Merge configs for TotalLoss
    # ------------------------------------------

    loss_cfg = copy.deepcopy(
        model_cfg
    )

    loss_cfg.update({

        "staged_activation":
        train_cfg[
            "staged_activation"
        ]
    })

    loss_fn = TotalLoss(
        loss_cfg
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=lr,

        weight_decay=1e-4
    )

    history = {

        "loss": [],

        "recon": [],

        "multires": [],

        "ortho": [],

        "grad": [],

        "content_kl": [],

        "speaker_kl": [],

        "environment_kl": [],

        "excitation_kl": [],

        "fidelity_kl": []
    }

    return (

        model,

        loss_fn,

        optimizer,

        subset_loader,

        history
    )

In [24]:
# ==================================================
# CELL 12
# One Epoch
# ==================================================

def train_epoch_subset(

    model,

    loader,

    loss_fn,

    optimizer,

    epoch,

    total_epochs
):

    model.train()

    loss_fn.current_epoch = epoch

    loss_fn.total_epochs = total_epochs

    running = {

        "loss": 0.0,

        "recon": 0.0,

        "multires": 0.0,

        "ortho": 0.0,

        "content_kl": 0.0,

        "speaker_kl": 0.0,

        "environment_kl": 0.0,

        "excitation_kl": 0.0,

        "fidelity_kl": 0.0
    }

    grad_total = 0

    n_batches = 0

    final_outputs = None
    final_batch = None

    for batch in loader:

        batch_gpu = {

            k:
            (
                v.to(device)

                if torch.is_tensor(v)

                else v
            )

            for k,v in batch.items()
        }

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(
            batch_gpu
        )

        loss_dict = loss_fn(

            outputs,

            batch_gpu
        )

        loss = loss_dict[
            "total"
        ]

        loss.backward()

        optimizer.step()

        grad_norm = 0

        for p in model.parameters():

            if p.grad is not None:

                grad_norm += (

                    p.grad.norm()
                    .item()
                )

        grad_total += grad_norm

        running["loss"] += float(
            loss.detach()
        )

        running["recon"] += float(
            loss_dict[
                "reconstruction"
            ].detach()
        )

        running["multires"] += float(
            loss_dict[
                "multires"
            ].detach()
        )

        running["ortho"] += float(
            loss_dict[
                "orthogonality"
            ].detach()
        )

        for latent in [

            "content",
            "speaker",
            "environment",
            "excitation",
            "fidelity"
        ]:

            running[
                f"{latent}_kl"
            ] += float(

                loss_dict[
                    f"{latent}_kl"
                ].detach()
            )

        final_outputs = outputs
        final_batch = batch_gpu

        n_batches += 1

    for k in running:

        running[k] /= n_batches

    grad_total /= n_batches

    return (

        running,

        grad_total,

        final_outputs,

        final_batch
    )

In [25]:
# ==================================================
# CELL 13
# Run Experiments
# ==================================================

all_results = {}

for n_samples in N_SAMPLES_LIST:

    print()
    print("=" * 80)
    print(
        f"RUNNING {n_samples} SAMPLE EXPERIMENT"
    )
    print("=" * 80)

    (
        model,
        loss_fn,
        optimizer,
        subset_loader,
        history

    ) = run_experiment(

        n_samples,
        LR
    )

    torch.cuda.empty_cache()
    gc.collect()

    progress = tqdm(

        range(EPOCHS),

        desc=f"{n_samples} Samples"
    )

    outputs = None
    batch_gpu = None

    for epoch in progress:

        (
            metrics,
            grad_norm,
            outputs,
            batch_gpu

        ) = train_epoch_subset(

            model,
            subset_loader,
            loss_fn,
            optimizer,
            epoch,
            EPOCHS
        )

        history["loss"].append(
            metrics["loss"]
        )

        history["recon"].append(
            metrics["recon"]
        )

        history["multires"].append(
            metrics["multires"]
        )

        history["ortho"].append(
            metrics["ortho"]
        )

        history["grad"].append(
            grad_norm
        )

        for latent in [

            "content",
            "speaker",
            "environment",
            "excitation",
            "fidelity"
        ]:

            history[
                f"{latent}_kl"
            ].append(

                metrics[
                    f"{latent}_kl"
                ]
            )

        if epoch % 10 == 0:

            progress.set_postfix({

                "loss":
                round(
                    metrics["loss"],
                    3
                ),

                "recon":
                round(
                    metrics["recon"],
                    3
                ),

                "grad":
                round(
                    grad_norm,
                    2
                )
            })

        gc.collect()

    target_lengths = (

        build_target_lengths(
            batch_gpu
        )
    )

    all_results[
        n_samples
    ] = {

        "model":
        model,

        "outputs":
        outputs,

        "batch":
        batch_gpu,

        "target_lengths":
        target_lengths,

        "history":
        history
    }

    print()

    print(
        f"{n_samples} SAMPLE RUN COMPLETE"
    )

    print(
        "Initial Loss:",
        round(
            history["loss"][0],
            4
        )
    )

    print(
        "Final Loss:",
        round(
            history["loss"][-1],
            4
        )
    )

    print(
        "Reduction:",
        round(
            history["loss"][0]
            -
            history["loss"][-1],
            4
        )
    )

    print(
        "Final Content KL:",
        round(
            history["content_kl"][-1],
            4
        )
    )

    print(
        "Final Environment KL:",
        round(
            history["environment_kl"][-1],
            4
        )
    )

    print(
        "Final Fidelity KL:",
        round(
            history["fidelity_kl"][-1],
            4
        )
    )

    print(
        "Final Orthogonality:",
        round(
            history["ortho"][-1],
            4
        )
    )

    print()

    print(
        "GPU Allocated:",
        round(
            torch.cuda.memory_allocated()
            /
            1024**3,
            2
        ),
        "GB"
    )

    torch.cuda.empty_cache()
    gc.collect()

print()
print("=" * 80)
print("ALL EXPERIMENTS COMPLETE")
print("=" * 80)

print(
    "Stored Results:",
    list(
        all_results.keys()
    )
)


RUNNING 4 SAMPLE EXPERIMENT


4 Samples:   0%|          | 0/100 [00:00<?, ?it/s]

: 